In [172]:
import pandas as pd

In [173]:
df_reqs = pd.read_csv('../data/challenge_14_requirements.csv')

df_reqs.head()

,Store,Item,Required
0,A,1,47
1,A,2,29
2,A,3,50
3,A,4,12
4,A,5,19


In [174]:
df_stores = pd.read_csv('../data/challenge_14_stores.csv')

df_stores.head()

,Store,Priority,City,State,Lon,Lat
0,A,26,Houston,TX,-95.562207,29.839187
1,B,4,Birmingham,AL,-86.625591,33.410321
2,C,25,Seattle,WA,-122.291399,47.603993
3,D,3,Detroit,MI,-83.255088,42.412144
4,E,5,Nashville,TN,-86.729174,36.135464


In [175]:
df_supply = pd.read_csv('../data/challenge_14_supply.csv')

df_supply

,Warehouse,Item,Count
0,Main,1,824
1,Main,2,1145
2,Main,3,1354
3,Main,4,916
4,Main,5,1120
5,Main,6,635
6,Main,7,1056
7,Main,8,824
8,Main,9,616
9,Main,10,679


In [176]:
# join dfs together and sort according to item and priority

df = pd.merge(df_stores, df_reqs, how='inner', on='Store')
df = pd.merge(df, df_supply, how='inner', on='Item').rename(columns={'Count': 'Supply'}).sort_values(['Item', 'Priority'])

df.head()

,Store,Priority,City,State,Lon,Lat,Item,Required,Warehouse,Supply
56,H,1,Fayetteville,NC,-78.907738,35.024302,1,11,Main,824
37,F,2,Chicago,IL,-87.597287,41.734000,1,71,Main,824
30,E,5,Nashville,TN,-86.729174,36.135464,1,44,Main,824
89,L,6,Los Angeles,CA,-118.386826,33.945722,1,26,Main,824
114,O,8,Dallas,TX,-96.770293,32.846730,1,15,Main,824


In [ ]:
# create the running sum of items allocated to each store and use it to determine how many items were allocated to each individual store

df['Running Sum'] = df.groupby(['Item'])['Required'].cumsum()
df = df.sort_values(['Item', 'Priority']).reset_index(drop=True)

for i in range(len(df)):
    if df.loc[i, 'Running Sum'] > df.loc[i, 'Supply']:
        df.loc[i, 'Running Sum'] = df.loc[i, 'Supply']
    try:
        if (df.loc[i, 'Item'] == df.loc[i-1, 'Item']):
            df.loc[i, 'Assigned'] = df.loc[i, 'Running Sum'] - df.loc[i-1, 'Running Sum']
    except KeyError:
        df.loc[i, 'Assigned'] = df.loc[i, 'Running Sum']

df.sort_values(['Store', 'Item'])

,Store,Priority,City,State,Lon,Lat,Item,Required,Warehouse,Supply,Running Sum,Assigned
17,A,26,Houston,TX,-95.562207,29.839187,1,47,Main,824,751,47.0
42,A,26,Houston,TX,-95.562207,29.839187,2,29,Main,1145,857,29.0
60,A,26,Houston,TX,-95.562207,29.839187,3,50,Main,1354,692,50.0
84,A,26,Houston,TX,-95.562207,29.839187,4,12,Main,916,757,12.0
107,A,26,Houston,TX,-95.562207,29.839187,5,19,Main,1120,1010,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...
120,Z,17,Las Vegas,NV,-115.242869,36.177034,6,18,Main,635,431,18.0
145,Z,17,Las Vegas,NV,-115.242869,36.177034,7,18,Main,1056,652,18.0
169,Z,17,Las Vegas,NV,-115.242869,36.177034,8,25,Main,824,550,25.0
187,Z,17,Las Vegas,NV,-115.242869,36.177034,9,2,Main,616,456,2.0
